# Perplejidad y longitud de contexto

Notebook opcional del **módulo 7** del curso (en la app: `uv run dev` y la ruta `#/modulo/contexto-largo`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

nombre = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(nombre)
modelo = AutoModelForCausalLM.from_pretrained(nombre, torch_dtype=torch.float32)
modelo.eval()

La perplejidad de una frase esperada frente a una sorprendente.

In [ ]:
def perplejidad(texto):
    x = tok(texto, return_tensors="pt")
    with torch.no_grad():
        perdida = modelo(**x, labels=x["input_ids"]).loss
    return torch.exp(perdida).item()

print(perplejidad("Apple Inc. is an American multinational technology company headquartered in Cupertino, California."))
print(perplejidad("A Falcon is a generative transformer model and it can't fly."))

Perplejidad con **ventana deslizante** sobre un texto largo, para distintas longitudes de contexto.

In [ ]:
texto = " ".join(["The history of transformers began with attention mechanisms for translation."] * 200)
ids = tok(texto, return_tensors="pt")["input_ids"]
for ventana in [64, 256, 1024]:
    nll, n = 0.0, 0
    for i in range(0, min(ids.shape[1], 2048) - ventana, ventana // 2):
        trozo = ids[:, i:i + ventana]
        objetivo = trozo.clone()
        objetivo[:, : ventana // 2] = -100          # solo se evalúa la segunda mitad (con contexto previo)
        with torch.no_grad():
            nll += modelo(trozo, labels=objetivo).loss.item()
        n += 1
    print(ventana, round(float(torch.exp(torch.tensor(nll / n))), 3))

En la configuración del modelo verás `rope_theta` y, si se activa, `rope_scaling` (por ejemplo `{"rope_type": "yarn", "factor": 4.0}`).

In [ ]:
print(modelo.config.rope_theta, getattr(modelo.config, "rope_scaling", None), modelo.config.max_position_embeddings)